# Vector-HaSH

In [10]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")  # 작은 matvec/pinv 다수 호출 시 BLAS 스레드 스폰 오버헤드로 10배+ 느려지는 문제 방지 (numpy import 전에 설정해야 함)
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 200  # 모든 figure에 공통 적용, compact하게

##### 드래그 슬라이드 다른 인터페이스 추가

In [11]:
import gc

# 데모 하나 로드할 때 이전 데모를 닫아서 메모리 회수하는 공용 인프라.
# ipywidgets 위젯은 화면에서 안 보이게 지워도(clear_output) Widget.widgets 전역
# registry에 계속 살아있어서 메모리가 안 풀림 -- .close()로 registry에서 빼야
# 실제 회수됨(캐시 dict 등도 그제서야 GC 대상이 됨).
_loaded_demos = {}  # name -> {"btn":.., "out":.., "widgets":[...]}


def _close_widget_tree(w):
    for child in getattr(w, "children", ()):
        _close_widget_tree(child)
    try:
        w.close()
    except Exception:
        pass


def _register_demo(name, btn, out, extra_widgets):
    _loaded_demos[name] = {"btn": btn, "out": out, "widgets": extra_widgets}


def _unload_demo(name):
    info = _loaded_demos.pop(name, None)
    if info is None:
        return
    info["out"].clear_output()
    for w in info["widgets"]:
        _close_widget_tree(w)
    info["btn"].disabled = False
    gc.collect()


def _unload_other_demos(except_name):
    for name in list(_loaded_demos):
        if name != except_name:
            _unload_demo(name)

## 1. Item Memory
#### - Capacity Test: Reconstruction Rate (Sensory Count vs Hippocampus Size)
#### - Noise Test: Noisy Input -> Cleanup -> Reconstruction

In [ ]:
import config as cfg

# 덮어쓸 파라미터로 새로운 설정 객체 생성 후 전역 설정 덮어쓰기
new_scaffold = cfg.ScaffoldConfig(
    module_periods=[3, 4, 5], Nh=400,
    connection_prob=0.6, threshold=0.5, nonlinearity="relu_threshold"
)
cfg.DEFAULT_SCAFFOLD = new_scaffold

%matplotlib inline
from experiments.experiment_item_capacity import (
    prepare_sensory_data,
    get_mem_for_Nh_sweep, render_node_states_panel, _stepper_row,
)
import ipywidgets as widgets
from IPython.display import display


def _load_2a():
    sbook_flattened = prepare_sensory_data()

    _mem_cache = {}  # (Nh, n_items_sub) -> (mem_sub, items_sub), Nh/n_items 안 바뀌면 재사용 (learn+fit_wgh 비쌈)

    def update_2a(Nh, n_items_sub, target_idx, noise_type, noise_ratio):
        key = (Nh, n_items_sub)
        if key not in _mem_cache:
            _mem_cache.clear()  # 캐시 하나만 유지 (메모리 아끼기)
            _mem_cache[key] = get_mem_for_Nh_sweep(sbook_flattened, Nh=Nh, n_items=n_items_sub)
        mem_sub, items_sub = _mem_cache[key]
        target_idx = min(target_idx, n_items_sub - 1)  # n_items_sub보다 커지면 잘라냄
        cmp = "<" if n_items_sub < Nh else ("=" if n_items_sub == Nh else ">")  # P vs Nh 실제 관계 그대로 표시
        render_node_states_panel(mem_sub, items_sub, target_idx, noise_type, noise_ratio,
                                  exp_label=f"P={n_items_sub} {cmp} Nh={Nh}")

    _style = {"description_width": "170px"}  # 라벨 길이 달라도 슬라이더 시작 위치/길이는 다 같게 고정폭
    Nh_slider = widgets.IntSlider(value=cfg.DEFAULT_SCAFFOLD.Nh, min=200, max=800, step=10,
                                   description="N_h:", style=_style, continuous_update=False)  # 학습(캐시미스)이 비싸서 드래그 중엔 값만 옮기고 뗄 때 1번만 재계산
    n_items_slider = widgets.IntSlider(value=cfg.DEFAULT_SCAFFOLD.Nh // 2 + 1, min=1, max=1000, step=10,
                                        description="N_s:", style=_style, continuous_update=False)  # 위와 동일 이유
    idx_slider = widgets.IntSlider(value=0, min=0, max=n_items_slider.value - 1, description="Item index:",
                                    style=_style, continuous_update=True)
    noise_dropdown = widgets.Dropdown(options=["masking", "salt_and_pepper"], value="salt_and_pepper",
                                       description="Noise type:", style=_style,
                                       layout=widgets.Layout(width="320px"))
    ratio_slider = widgets.FloatSlider(value=0.1, min=0, max=0.9, step=0.1, description="Noise ratio:",
                                        style=_style, continuous_update=True)

    def _sync_idx_max(change):
        idx_slider.max = max(0, change["new"] - 1)   # n_items_sub 바뀌면 item index 최대치도 같이 조절
    n_items_slider.observe(_sync_idx_max, names="value")

    out = widgets.interactive_output(update_2a, {
        "Nh": Nh_slider, "n_items_sub": n_items_slider, "target_idx": idx_slider,
        "noise_type": noise_dropdown, "noise_ratio": ratio_slider,
    })
    # display()에 인자를 여러 개 넘기면 각각 별도 output area로 렌더링돼서
    # 사이사이 여백이 큼 -> VBox 하나로 묶어서 한 번만 display (다른 섹션과 간격 통일)
    # 변수에 담아두는 이유: 데모 전환 시 _close_widget_tree로 재귀적으로 닫으려면
    # 최상위 컨테이너를 참조하고 있어야 함 (display()에 바로 넘기면 참조가 안 남음)
    _panel = widgets.VBox([
        _stepper_row(widgets, Nh_slider), _stepper_row(widgets, n_items_slider),
        _stepper_row(widgets, idx_slider, offset=1),
        noise_dropdown, _stepper_row(widgets, ratio_slider),
    ])
    display(_panel, out)
    return [_panel, out]


# 메모리 아끼려고 버튼 누를 때만 로드 (데이터셋+캐시가 무거움, 4개 데모 동시 로드하면 512MB 서버에서 OOM)
# 다른 데모가 로드돼 있으면 먼저 닫아서(위젯 close + 캐시 GC) 메모리 확보
_btn_2a = widgets.Button(description="Load demo 2a", button_style="info", icon="play")
_out_2a = widgets.Output()

def _on_click_2a(_):
    _unload_other_demos("2a")
    _btn_2a.disabled = True
    with _out_2a:
        created = _load_2a()
    _register_demo("2a", _btn_2a, _out_2a, created)
_btn_2a.on_click(_on_click_2a)
display(_btn_2a, _out_2a)


####### 복원을 못하는 경우 or 복원했는데 다른 이미지인 경우 어떤 grid state가 나오고 그게 원본 grid 로 부터 얼마나 차이가 있는지 확인
####### clean up 몇번 해야 복원 되고 하는지
####### 이미지로 복원되지 않은 경우 1. 200개의 sensory 가 아닌 3400개에서 나오는건지, onehot 이 아닌 multihot 처럼 3600개가 아닌 조합에서 나오는건지 확인.
####### cleanup 과정에서 어떻게 복원되는지 확인

####### cleanup 이라 함은 WTA (grid state 교정) -> hpc
####### 복원 했을 때 다른 이미지가 나왔다면 몇번째 이미지인지 추가
####### module diff 계산시에도 모듈 순서 고려해서 나오게 하기 (one-hot 상에서 bit diff)
####### cleanup 되기 전에(WTA 되기 전에) 모듈 상태도 보여주기 (실수 형태 나올것으로 예상)
####### module 시각화할때 마름모처럼 나타내기, shear transfrom, 중간에 구분선 그리기



####### Wgh 에서 g가 one-hot encoding 된 상태인지 파악

Button(button_style='info', description='Load demo 2a', icon='play', style=ButtonStyle())

Output()

## 2. Spatial Memory
#### - Path Integration: Sensory Reconstruction on Novel Paths
#### - Spatial Evaluation: Revisited vs. Unvisited Areas (Near vs. Far)

In [13]:
import config as cfg

# 덮어쓸 파라미터로 새로운 설정 객체 생성 후 전역 설정 덮어쓰기
new_scaffold = cfg.ScaffoldConfig(
    module_periods=[3, 4, 5], Nh=400,
    connection_prob=0.6, threshold=0.5, nonlinearity="relu_threshold"
)
cfg.DEFAULT_SCAFFOLD = new_scaffold

from experiments.experiment_spatial_navigation import (
    build_fig4c_demo, build_novel_trajectory,
    demo_revisit_predictions, demo_unvisited_predictions, demo_unvisited_by_distance,
    plot_unvisited_distance_map, demo_sensory_to_location,
)
from experiments.experiment_item_capacity import _stepper_row
import ipywidgets as widgets
from IPython.display import display


def _load_3a():
    # trained_length = 원래(학습한) 경로 길이, novel_length = 새 경로 길이,
    # n_overlap = 새 경로가 원래 경로를 몇 번 재방문하도록 강제할지 (=재방문 수)
    _model_cache = {}      # trained_length -> model (build_fig4c_demo, 제일 비쌈)
    _novel_cache = {}      # (trained_length, novel_length, n_overlap) -> novel_model

    def update_3a(trained_length, novel_length, n_overlap):
        if trained_length not in _model_cache:
            _model_cache.clear()
            _novel_cache.clear()  # model 바뀌면 novel_model도 다시 만들어야 함
            _model_cache[trained_length] = build_fig4c_demo(trained_length=trained_length, seed=3)
        model = _model_cache[trained_length]

        novel_key = (trained_length, novel_length, n_overlap)
        if novel_key not in _novel_cache:
            _novel_cache.clear()
            _novel_cache[novel_key] = build_novel_trajectory(
                model, novel_length=novel_length, n_overlap=n_overlap, seed=1)
        novel_model = _novel_cache[novel_key]

        # 1. 원래 경로 vs 새 경로 시각화 (강제 재방문 지점은 별표로 표시)
        unvisited = plot_unvisited_distance_map(model, novel_model, n_show=4)

        # 2. 새 경로가 원래 경로를 재방문하는 지점들에서 sensory 예측이 True와 얼마나 맞는지
        demo_revisit_predictions(model, novel_model, n_revisits=n_overlap)

        # 2c. 미방문 지점을 학습된 경로까지 거리(가까운 노드 vs 먼 노드)로 나눠서 비교
        #     -- 가까울수록(재방문 지점에 가까울수록) recall이 더 정확하고, 멀수록
        #     거의 무작위에 가까워지는 것을 확인
        demo_unvisited_by_distance(model, novel_model, unvisited)

    _style = {"description_width": "170px"}  # 고정폭: 라벨 안 잘리면서 슬라이더 시작 위치/길이 다 같게
    trained_length_slider = widgets.IntSlider(value=100, min=20, max=300, step=10,
                                               description="Original path length:", style=_style, continuous_update=False)
    novel_length_slider = widgets.IntSlider(value=100, min=20, max=300, step=10,
                                             description="New path length:", style=_style, continuous_update=False)
    n_overlap_slider = widgets.IntSlider(value=4, min=1, max=10, step=1,
                                          description="# revisits:", style=_style, continuous_update=False)

    out = widgets.interactive_output(update_3a, {
        "trained_length": trained_length_slider, "novel_length": novel_length_slider,
        "n_overlap": n_overlap_slider,
    })
    # 변수에 담아두는 이유: 데모 전환 시 _close_widget_tree로 재귀적으로 닫으려면
    # 최상위 컨테이너를 참조하고 있어야 함 (display()에 바로 넘기면 참조가 안 남음)
    _panel = widgets.VBox([
        _stepper_row(widgets, trained_length_slider), _stepper_row(widgets, novel_length_slider),
        _stepper_row(widgets, n_overlap_slider),
    ])
    display(_panel, out)
    return [_panel, out]


# 메모리 아끼려고 버튼 누를 때만 로드 (모델 캐시가 무거움, 4개 데모 동시 로드하면 512MB 서버에서 OOM)
# 다른 데모가 로드돼 있으면 먼저 닫아서(위젯 close + 캐시 GC) 메모리 확보
_btn_3a = widgets.Button(description="Load demo 3a", button_style="info", icon="play")
_out_3a = widgets.Output()

def _on_click_3a(_):
    _unload_other_demos("3a")
    _btn_3a.disabled = True
    with _out_3a:
        created = _load_3a()
    _register_demo("3a", _btn_3a, _out_3a, created)
_btn_3a.on_click(_on_click_3a)
display(_btn_3a, _out_3a)

# 2b. 새 경로가 원래 경로와 한 번도 안 겹친(학습 때 결합이 없었던) 지점들에서는
#     sensory 예측이 어떻게 나오는지 -- 재방문 지점(cos_sim ~1.0)과 비교해보면
#     아직 안 배운 위치라 recall이 훨씬 부정확함을 확인 가능
# demo_unvisited_predictions(model, novel_model, n_show=10)

# 3. sensory 입력 -> 어느 위치인지 역추론
# demo_sensory_to_location(model, query_step=5)
########### grid state 색칠된 크기는 같게, nxn의 사이즈는 다르게
########### 위 실험도 grid state 확인해보기,hpc
########### 인접한 grid state의 hpc 가 다를것(확인해보기)

Button(button_style='info', description='Load demo 3a', icon='play', style=ButtonStyle())

Output()

In [5]:
########## 마커 작게, 지도 하나만, position이랑 grid state 매핑 어떻게 되어 있는지 위랑 차이 있는지

## 3. Memory Palace

#### 3a. Reconstruction Beyond Hippocampus Capacity (Items > $N_h$)

In [ ]:
import config as cfg

# 덮어쓸 파라미터로 새로운 설정 객체 생성 후 전역 설정 덮어쓰기
new_scaffold = cfg.ScaffoldConfig(
    module_periods=[3, 4, 5], Nh=400,
    connection_prob=0.6, threshold=0.5, nonlinearity="relu_threshold"
)
cfg.DEFAULT_SCAFFOLD = new_scaffold

from experiments.experiment_memory_palace import (
    build_seq_scaffold, make_embedded_image_book_for_fig7,
    make_hairpin_path, path_to_indices,
    recall_sequence_once, plot_palace_path, cos_sim,
)
from experiments.experiment_item_capacity import _stepper_row
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def load_number_card_images_grayscale(numbers):
    """number_card_60x60/ 폴더(1~999, 미리 렌더링해둔 PNG)에서 numbers에 해당하는
    카드만 그레이스케일로 불러와 (Ns, len(numbers)) 형태로 반환."""
    from PIL import Image
    folder = os.path.abspath(os.path.join(os.getcwd(), "..", "number_card_60x60"))
    imgs = [np.array(Image.open(os.path.join(folder, f"{n:03d}.png")).convert("L"), dtype=np.float64)
            for n in numbers]
    arr = np.stack(imgs, axis=0)  # (len(numbers), 60, 60)
    return arr.reshape(len(numbers), -1).T  # (Ns, len(numbers))


def make_numbered_card_book(
    Ns, Nstates, Npos,
    block_x0=0, block_y0=0, block_w=13, block_h=8,
    seed=0, use_tanh_inverse=True,
):
    """트럼프 카드(52장 제한) 대신, block_w*block_h개 위치마다 1..n_positions 숫자
    카드를 그때그때 새로 렌더링해서 중복 없이 랜덤 순서로 배치한다. 카드가 전부
    유일하므로 반복/워터마크 없이도 pinv가 항상 full rank."""
    rng = np.random.default_rng(seed)
    img_h = img_w = int(round(np.sqrt(Ns)))
    assert img_h * img_w == Ns, f"Ns({Ns})는 정사각형 이미지 픽셀 수여야 합니다."

    n_positions = block_w * block_h
    assert n_positions <= 9999, "number_card_60x60/ 폴더에 1~9999까지만 미리 렌더링해둠"
    numbers = rng.permutation(n_positions) + 1  # 1..n_positions, 중복 없이 섞음
    # ponytail: float32로 낮춰서 메모리 절반 (Render 512MB 한도, 데모용이라 정밀도 손실 무해)
    img_flat = load_number_card_images_grayscale(numbers).astype(np.float32)
    # 배경이 전부 흰색이라 카드끼리 거의 동일 -> arctanh 포화까지 겹치면 pinv rank 붕괴.
    # 카드별로 고유한 배경 텍스처 노이즈를 살짝 섞어서 서로 잘 구분되게(=well-conditioned) 만듦.
    img_flat += rng.standard_normal(img_flat.shape).astype(np.float32) * 10.0
    img_flat -= img_flat.mean()

    if use_tanh_inverse:
        smin, smax = np.amin(img_flat), np.amax(img_flat)
        scale = 1.9 / (smax - smin)
        shift = -0.95 - smin * scale
        img_flat *= scale
        img_flat += shift  # np.interp((smin,smax)->(-0.95,0.95))와 동일한 선형 변환, in-place
        np.arctanh(img_flat, out=img_flat)
        img_embed = img_flat
    else:
        img_embed = np.sign(img_flat)
        smin, smax = None, None

    # block이 (0,0)부터 시작해서 Nstates 전체를 정확히 덮는 경우(이 앱의 실제 사용
    # 패턴) x*Npos+y가 항상 k와 같은 순서로 증가 -> sbook_full은 img_embed와 동일.
    # (예전엔 랜덤 배열 만들고 한 칸씩 덮어썼는데 전부 버려지는 값이라 낭비였음)
    assert block_x0 == 0 and block_y0 == 0 and n_positions == Nstates and Npos == block_h
    sbook_full = img_embed
    return sbook_full, smin, smax


def plot_palace_and_recall(path, block_w, block_h, img_h, img_w, t, panels, show_map=True):
    """지도(위, 60x60 전체 격자 고정 뷰 + 현재 depth만큼 그려진 path + 현재 t 위치)와
    recall 이미지 3장(아래)을 fig 하나에 묶어서 그림. show_map=False면 지도 없이
    이미지 3장만. demo_4a_v2_interactive와 분리된 독립 함수라 path/이미지 크기만
    맞으면 다른 곳에서도 그대로 재사용 가능."""
    if show_map:
        fig = plt.figure(figsize=(8.5, 5))
        gs = fig.add_gridspec(2, len(panels), height_ratios=[1.2, 1])

        ax_map = fig.add_subplot(gs[0, :])
        ax_map.plot(path[:, 0], path[:, 1], "-o", markersize=1, linewidth=0.6, alpha=0.7, color="tab:blue")
        ax_map.plot(path[0, 0], path[0, 1], "gs", markersize=5, label="start (t=0)")
        ax_map.plot(path[-1, 0], path[-1, 1], "r^", markersize=5, label=f"end (t={len(path) - 1})")
        hx, hy = path[t]
        ax_map.plot(hx, hy, "*", color="orange", markersize=9, markeredgecolor="black",
                    label=f"current (t={t})", zorder=5)
        ax_map.set_xlim(-1, block_w)
        ax_map.set_ylim(-1, block_h)
        ax_map.set_xlabel("x"); ax_map.set_ylabel("y")
        ax_map.set_title(f"Palace trajectory ({block_w}x{block_h} grid, depth={len(path)})")
        ax_map.set_aspect("equal")
        ax_map.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0))

        panel_row = 1
    else:
        fig = plt.figure(figsize=(13, 3.4))
        gs = fig.add_gridspec(1, len(panels))
        panel_row = 0

    for i, (vec, title, sim) in enumerate(panels):
        ax = fig.add_subplot(gs[panel_row, i])
        ax.imshow(vec.reshape(img_h, img_w), cmap="gray")
        subtitle = f"cos={sim:.3f}" if sim is not None else " "
        ax.set_title(f"{title}\n{subtitle}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

    plt.tight_layout()
    plt.show()


def demo_4a_v2_interactive(
    lambdas=(4, 5, 7), Nh=40, gamma=0.6, thresh=0.5,
    Ns=3600, max_depth=3600, seed=0, show_map=True,
):
    """4a와 같은 구조, True/new item 소스만 교체:
    old item(True, Clean recall) = miniimagenet, new item = 숫자 카드(중복 없이 유일).
    max_depth는 palace 격자(60x60 상한)에 미리 채워두는 아이템 수의 상한이고,
    실제 시각화(지도/이미지)는 depth 슬라이더 값 기준으로 그 중 일부만 보여줌.
    Nh는 슬라이더로 바꿀 수 있고, 바뀔 때만 scaffold를 다시 학습(캐시)함."""
    img_h = img_w = int(round(np.sqrt(Ns)))

    Npos = int(np.prod(lambdas))
    block_w = block_h = min(60, Npos)  # palace 격자 60x60(=3600칸) 상한
    n_cards = max_depth
    assert max_depth <= block_w * block_h, (
        f"max_depth={max_depth}가 {block_w}x{block_h}={block_w * block_h}칸을 넘음."
    )
    Nstates = Npos * Npos  # scaf 없이도 lambdas만으로 정해지는 값

    _scaf_cache = {}
    def get_scaf(Nh_val):
        if Nh_val not in _scaf_cache:
            _scaf_cache.clear()  # 캐시 하나만 유지 (메모리 아끼기)
            s = build_seq_scaffold(list(lambdas), Nh_val, gamma=gamma, thresh=thresh, nruns=1)
            assert s["Npos"] == Npos
            _scaf_cache[Nh_val] = s
        return _scaf_cache[Nh_val]

    # old item = miniimagenet (기존 4a의 new item 자리에 있던 소스), Nh와 무관
    sbook_old, _, _ = make_embedded_image_book_for_fig7(
        Ns, Nstates, Npos, 0, 0, block_w, block_h,
        seed=seed, shuffle_images=False, use_tanh_inverse=True
    )
    # new item = 숫자 카드, depth만큼 유일하게 생성 (기존 4a의 old item 자리에 있던 소스), Nh와 무관
    mbook_new, _, _ = make_numbered_card_book(
        Ns, Nstates, Npos, 0, 0, block_w, block_h,
        seed=seed, use_tanh_inverse=True
    )

    path_all = make_hairpin_path(block_w, block_h, 0, 0)[:max_depth]
    idxs_all = path_to_indices(path_all, Npos)

    _recall_cache = {}
    def get_recall(Nh_val, depth):
        # (Nh_val, depth) 안 바뀌면 재사용 (recall_sequence_once + pinv 비쌈).
        # t만 바뀔 때는(2a/3a와 같은 패턴) 여기 안 타고 캐시된 결과에서 슬라이스만 함.
        key = (Nh_val, depth)
        if key not in _recall_cache:
            _recall_cache.clear()  # 캐시 하나만 유지 (메모리 아끼기)
            scaf = get_scaf(Nh_val)
            idxs_seq = idxs_all[:depth]
            P_seq = scaf["pbook_flat"][:, :, idxs_seq]
            S_seq = sbook_old[:, idxs_seq]
            M_new = mbook_new[:, idxs_seq]

            S_clean = recall_sequence_once(scaf, S_seq, P_seq, depth, np.random.default_rng(1))

            S_addr1 = np.sign(S_clean[0])
            Wms = M_new @ np.linalg.pinv(S_addr1)
            M_rec_clean = Wms @ S_addr1
            _recall_cache[key] = (S_clean, M_rec_clean)
        return _recall_cache[key]

    def update_plot(Nh_val, depth, t):
        if t >= depth:
            return

        S_clean, M_rec_clean = get_recall(Nh_val, depth)

        true_s = sbook_old[:, idxs_all[t]]
        true_m = mbook_new[:, idxs_all[t]]

        idx_label = t  # 격자 flat 주소 대신 방문 순서(1,2,3...와 대응하는 0-index)로 표시
        panels = [
            (true_s, f"Stored item #{idx_label + 1}", None),
            (S_clean[0, :, t],
             f"Recalled item #{idx_label + 1} (cos_sim={cos_sim(S_clean[0, :, t], true_s):.3f})", None),
            (true_m, "Mnemonic item", None),
            (M_rec_clean[:, t],
             f"Recalled mnemonic item (cos_sim={cos_sim(M_rec_clean[:, t], true_m):.3f})", None),
        ]

        plot_palace_and_recall(path_all[:depth], block_w, block_h, img_h, img_w, t, panels, show_map=show_map)

    _style = {"description_width": "170px"}  # 라벨 길이 달라도 슬라이더 시작 위치/길이는 다 같게 고정폭
    Nh_slider = widgets.IntSlider(value=Nh, min=10, max=800, step=5,
                                   description="N_h:", style=_style, continuous_update=False)
    depth_slider = widgets.IntSlider(value=min(13, n_cards), min=2, max=n_cards, step=1,
                                      description="N_s:", style=_style, continuous_update=False)
    t_slider = widgets.IntSlider(value=0, min=0, max=depth_slider.value - 1, step=1,
                                  description="Item index:", style=_style, continuous_update=False)

    def _sync_t_max(change):
        t_slider.max = max(0, change["new"] - 1)  # t는 depth를 못 넘게 자동 조절
        if t_slider.value > t_slider.max:
            t_slider.value = t_slider.max
    depth_slider.observe(_sync_t_max, names="value")

    out = widgets.interactive_output(update_plot, {
        "Nh_val": Nh_slider, "depth": depth_slider, "t": t_slider,
    })
    # 변수에 담아두는 이유: 데모 전환 시 _close_widget_tree로 재귀적으로 닫으려면
    # 최상위 컨테이너를 참조하고 있어야 함 (display()에 바로 넘기면 참조가 안 남음)
    _panel = widgets.VBox([
        _stepper_row(widgets, Nh_slider), _stepper_row(widgets, depth_slider),
        _stepper_row(widgets, t_slider, offset=1),
    ])
    display(_panel, out)
    return [_panel, out]


# 메모리 아끼려고 버튼 누를 때만 로드 (miniimagenet+숫자카드+scaffold가 무거움, 4개 데모 동시 로드하면 512MB 서버에서 OOM)
# 다른 데모가 로드돼 있으면 먼저 닫아서(위젯 close + 캐시 GC) 메모리 확보
import ipywidgets as widgets
from IPython.display import display

_btn_4a = widgets.Button(description="Load demo 4a", button_style="info", icon="play")
_out_4a = widgets.Output()

def _on_click_4a(_):
    _unload_other_demos("4a")
    _btn_4a.disabled = True
    with _out_4a:
        created = demo_4a_v2_interactive(
            lambdas=(2, 3, 5), Nh=200, gamma=0.6, thresh=0.5, Ns=3600,
            max_depth=900, show_map=False,
        )
    _register_demo("4a", _btn_4a, _out_4a, created)
_btn_4a.on_click(_on_click_4a)
display(_btn_4a, _out_4a)

Button(button_style='info', description='Load demo 4a', icon='play', style=ButtonStyle())

Output()

#### 3b. Cleanup Test: Reconstruction of Original Novel Items from Noisy Inputs

In [ ]:
import config as cfg

# 덮어쓸 파라미터로 새로운 설정 객체 생성 후 전역 설정 덮어쓰기
new_scaffold = cfg.ScaffoldConfig(
    module_periods=[2, 3, 5], Nh=400,
    connection_prob=0.6, threshold=0.5, nonlinearity="relu_threshold"
)
cfg.DEFAULT_SCAFFOLD = new_scaffold

import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from experiments.experiment_memory_palace import (
    build_seq_scaffold, make_hairpin_path, path_to_indices,
    make_embedded_image_book_for_fig7,
    recall_sequence_once, cos_sim,
)
from experiments.experiment_item_capacity import apply_noise, _stepper_row


def load_number_card_images_grayscale(numbers):
    """number_card_60x60/ 폴더(1~999, 미리 렌더링해둔 PNG)에서 numbers에 해당하는
    카드만 그레이스케일로 불러와 (Ns, len(numbers)) 형태로 반환."""
    from PIL import Image
    folder = os.path.abspath(os.path.join(os.getcwd(), "..", "number_card_60x60"))
    imgs = [np.array(Image.open(os.path.join(folder, f"{n:03d}.png")).convert("L"), dtype=np.float64)
            for n in numbers]
    arr = np.stack(imgs, axis=0)  # (len(numbers), 60, 60)
    return arr.reshape(len(numbers), -1).T  # (Ns, len(numbers))


def make_numbered_card_book(
    Ns, Nstates, Npos,
    block_x0=0, block_y0=0, block_w=13, block_h=8,
    seed=0, use_tanh_inverse=True,
):
    """4a-v2와 동일: 트럼프 카드(52장 제한) 대신 block_w*block_h개 위치마다
    1..n_positions 숫자 카드를 그때그때 새로 렌더링해서 중복 없이 랜덤 순서로
    배치한다. 카드가 전부 유일하므로 반복/워터마크 없이도 pinv가 항상 full rank."""
    rng = np.random.default_rng(seed)
    img_h = img_w = int(round(np.sqrt(Ns)))
    assert img_h * img_w == Ns, f"Ns({Ns})는 정사각형 이미지 픽셀 수여야 합니다."

    n_positions = block_w * block_h
    assert n_positions <= 9999, "number_card_60x60/ 폴더에 1~9999까지만 미리 렌더링해둠"
    numbers = rng.permutation(n_positions) + 1  # 1..n_positions, 중복 없이 섞음
    # ponytail: float32로 낮춰서 메모리 절반 (Render 512MB 한도, 데모용이라 정밀도 손실 무해)
    img_flat = load_number_card_images_grayscale(numbers).astype(np.float32)
    # 배경이 전부 흰색이라 카드끼리 거의 동일 -> arctanh 포화까지 겹치면 pinv rank 붕괴.
    # 카드별로 고유한 배경 텍스처 노이즈를 살짝 섞어서 서로 잘 구분되게(=well-conditioned) 만듦.
    img_flat += rng.standard_normal(img_flat.shape).astype(np.float32) * 10.0
    img_flat -= img_flat.mean()

    if use_tanh_inverse:
        smin, smax = np.amin(img_flat), np.amax(img_flat)
        scale = 1.9 / (smax - smin)
        shift = -0.95 - smin * scale
        img_flat *= scale
        img_flat += shift  # np.interp((smin,smax)->(-0.95,0.95))와 동일한 선형 변환, in-place
        np.arctanh(img_flat, out=img_flat)
        img_embed = img_flat
    else:
        img_embed = np.sign(img_flat)
        smin, smax = None, None

    # block이 (0,0)부터 시작해서 Nstates 전체를 정확히 덮는 경우(이 앱의 실제 사용
    # 패턴) x*Npos+y가 항상 k와 같은 순서로 증가 -> sbook_full은 img_embed와 동일.
    # (예전엔 랜덤 배열 만들고 한 칸씩 덮어썼는데 전부 버려지는 값이라 낭비였음)
    assert block_x0 == 0 and block_y0 == 0 and n_positions == Nstates and Npos == block_h
    sbook_full = img_embed
    return sbook_full, smin, smax


def demo_4b_v2_interactive(
    lambdas=(4, 5, 7), Nh=40, gamma=0.6, thresh=0.5,
    Ns=3600, max_depth=3600, seed=0,
):
    """4b와 같은 3단계 파이프라인(Wsm_raw로 noisy item->noisy sensory->old item
    cleanup->Wms로 new item 재복원)이지만, old item(True sensory)/new item 소스는
    4a-v2와 동일하게 miniimagenet/숫자카드로 교체. 구조도 4a-v2와 맞춤:
    palace 격자 60x60 고정(max_depth는 그 안에서의 상한), Nh는 슬라이더로 바꿀 수
    있고 바뀔 때만 scaffold 재학습(캐시), depth/t도 (Nh,depth) 바뀔 때만
    recall 파이프라인 재계산(캐시)하고 t/noise만 바뀌면 캐시된 결과 재사용."""
    img_h = img_w = int(round(np.sqrt(Ns)))

    Npos = int(np.prod(lambdas))
    block_w = block_h = min(60, Npos)  # palace 격자 60x60(=3600칸) 상한
    n_cards = max_depth
    assert max_depth <= block_w * block_h, (
        f"max_depth={max_depth}가 {block_w}x{block_h}={block_w * block_h}칸을 넘음."
    )
    Nstates = Npos * Npos  # scaf 없이도 lambdas만으로 정해지는 값

    _scaf_cache = {}
    def get_scaf(Nh_val):
        if Nh_val not in _scaf_cache:
            _scaf_cache.clear()  # 캐시 하나만 유지 (메모리 아끼기)
            s = build_seq_scaffold(list(lambdas), Nh_val, gamma=gamma, thresh=thresh, nruns=1)
            assert s["Npos"] == Npos
            _scaf_cache[Nh_val] = s
        return _scaf_cache[Nh_val]

    # old item = miniimagenet (4b에서는 new item 자리였던 소스), Nh와 무관
    sbook_old, _, _ = make_embedded_image_book_for_fig7(
        Ns, Nstates, Npos, 0, 0, block_w, block_h,
        seed=seed, shuffle_images=False, use_tanh_inverse=True
    )
    # new item = 숫자 카드 (4b에서는 old item 자리였던 소스), Nh와 무관
    mbook_new, _, _ = make_numbered_card_book(
        Ns, Nstates, Npos, 0, 0, block_w, block_h,
        seed=seed, use_tanh_inverse=True
    )

    path_all = make_hairpin_path(block_w, block_h, 0, 0)[:max_depth]
    idxs_all = path_to_indices(path_all, Npos)

    _pipeline_cache = {}
    def get_pipeline(Nh_val, depth):
        # (Nh_val, depth) 안 바뀌면 재사용 (recall_sequence_once + pinv 비쌈).
        # t/noise만 바뀔 때는 여기 안 타고 캐시된 결과로 recover()만 다시 함.
        key = (Nh_val, depth)
        if key not in _pipeline_cache:
            _pipeline_cache.clear()  # 캐시 하나만 유지 (메모리 아끼기)
            scaf = get_scaf(Nh_val)
            idxs_seq = idxs_all[:depth]
            P_seq = scaf["pbook_flat"][:, :, idxs_seq]
            S_seq = sbook_old[:, idxs_seq]
            M_seq = mbook_new[:, idxs_seq]

            S_clean = recall_sequence_once(scaf, S_seq, P_seq, depth, np.random.default_rng(1))
            S_addr = np.sign(S_clean[0])
            Wms = M_seq @ np.linalg.pinv(S_addr)          # 주소 -> new item
            Wsm_raw = S_seq @ np.linalg.pinv(M_seq)       # new item -> sensory (원본 스케일)
            _pipeline_cache[key] = (scaf, S_seq, M_seq, P_seq, S_clean, Wms, Wsm_raw)
        return _pipeline_cache[key]

    def run_4b_v2(Nh_val, depth, t, noise_ratio_vis):
        if t >= depth:
            return

        scaf, S_seq, M_seq, P_seq, S_clean, Wms, Wsm_raw = get_pipeline(Nh_val, depth)

        def recover(noisy_item, tt):
            sensory_est_noisy = Wsm_raw @ noisy_item

            S_query = S_seq.copy()
            S_query[:, tt] = sensory_est_noisy
            S_rec = recall_sequence_once(scaf, S_seq, P_seq, depth, np.random.default_rng(1),
                                          S_query=S_query)
            sensory_cleaned = S_rec[0, :, tt]
            addr_clean = np.sign(sensory_cleaned)

            item_rec = Wms @ addr_clean
            return sensory_est_noisy, sensory_cleaned, item_rec

        def apply_noise_or_clean(vec, noise_ratio):
            return vec if noise_ratio == 0.0 else apply_noise(vec, "salt_and_pepper", noise_ratio, seed=2)

        true_sensory = S_seq[:, t]
        sensory_baseline_rec = S_clean[0, :, t]
        true_item = M_seq[:, t]
        noisy_item = apply_noise_or_clean(true_item, noise_ratio_vis)
        sensory_est_noisy, sensory_cleaned, item_rec = recover(noisy_item, t)

        idx_label = t  # 격자 flat 주소 대신 방문 순서(1,2,3...와 대응하는 0-index)로 표시
        panels = [
            (true_sensory, f"Stored item #{idx_label + 1}", None),
            (sensory_baseline_rec,
             f"Recalled item #{idx_label + 1} (cos_sim={cos_sim(sensory_baseline_rec, true_sensory):.3f})", None),
            (true_item, "Mnemonic item", None),
            (noisy_item, "Noisy item", None),
            (sensory_est_noisy,
             f"Noisy sensory recon (cos_sim={cos_sim(sensory_est_noisy, true_sensory):.3f})", None),
            (sensory_cleaned,
             f"Cleanup sensory recall (cos_sim={cos_sim(sensory_cleaned, true_sensory):.3f})", None),
            (item_rec, f"Recalled mnemonic item (cos_sim={cos_sim(item_rec, true_item):.3f})", None),
        ]
        fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 3.4))
        for ax, (vec, title, sim) in zip(axes, panels):
            ax.imshow(vec.reshape(img_h, img_w), cmap="gray")
            subtitle = f"cos={sim:.3f}" if sim is not None else " "
            ax.set_title(f"{title}\n{subtitle}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        plt.show()

    _style = {"description_width": "170px"}  # 라벨 길이 달라도 슬라이더 시작 위치/길이는 다 같게 고정폭
    Nh_slider = widgets.IntSlider(value=Nh, min=10, max=400, step=5,
                                   description="N_h:", style=_style, continuous_update=False)
    depth_slider = widgets.IntSlider(value=min(30, n_cards), min=2, max=n_cards, step=1,
                                      description="N_s:", style=_style, continuous_update=False)
    t_slider = widgets.IntSlider(value=0, min=0, max=depth_slider.value - 1, step=1,
                                  description="Item index:", style=_style, continuous_update=False)
    noise_slider = widgets.FloatSlider(value=0.3, min=0.0, max=0.9, step=0.1,
                                        description="Noise ratio:", style=_style, continuous_update=False)

    def _sync_t_max(change):
        t_slider.max = max(0, change["new"] - 1)  # t는 depth를 못 넘게 자동 조절
        if t_slider.value > t_slider.max:
            t_slider.value = t_slider.max
    depth_slider.observe(_sync_t_max, names="value")

    out = widgets.interactive_output(run_4b_v2, {
        "Nh_val": Nh_slider, "depth": depth_slider, "t": t_slider, "noise_ratio_vis": noise_slider,
    })
    # 변수에 담아두는 이유: 데모 전환 시 _close_widget_tree로 재귀적으로 닫으려면
    # 최상위 컨테이너를 참조하고 있어야 함 (display()에 바로 넘기면 참조가 안 남음)
    _panel = widgets.VBox([
        _stepper_row(widgets, Nh_slider), _stepper_row(widgets, depth_slider),
        _stepper_row(widgets, t_slider, offset=1), _stepper_row(widgets, noise_slider),
    ])
    display(_panel, out)
    return [_panel, out]


# 메모리 아끼려고 버튼 누를 때만 로드 (miniimagenet+숫자카드+scaffold가 무거움, 4개 데모 동시 로드하면 512MB 서버에서 OOM)
# 다른 데모가 로드돼 있으면 먼저 닫아서(위젯 close + 캐시 GC) 메모리 확보
_btn_4b = widgets.Button(description="Load demo 4b", button_style="info", icon="play")
_out_4b = widgets.Output()

def _on_click_4b(_):
    _unload_other_demos("4b")
    _btn_4b.disabled = True
    with _out_4b:
        created = demo_4b_v2_interactive(
            lambdas=(2, 3, 5), Nh=200, gamma=0.6, thresh=0.5, Ns=3600,
            max_depth=900,
        )
    _register_demo("4b", _btn_4b, _out_4b, created)
_btn_4b.on_click(_on_click_4b)
display(_btn_4b, _out_4b)

Button(button_style='info', description='Load demo 4b', icon='play', style=ButtonStyle())

Output()